# 📖 Notebook 1: Job Queue Design & Priority Scheduling

In this notebook we build a **priority job queue** from scratch using Redis sorted sets. By the end you'll understand how job schedulers decide *which job runs next* and how to prevent low-priority jobs from starving.

## Learning Objectives

- Understand the difference between a Task, a Job, and an Execution
- Build a priority queue with Redis sorted sets (ZSET)
- Implement fair scheduling across priority levels
- Prevent starvation of low-priority jobs

## 🛠️ Setup

Start the infrastructure first:

```bash
cd 06-system-designs/job-scheduler
docker compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `job_scheduler`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import redis
import json
import time
import uuid
from datetime import datetime, timedelta

# Database connection
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "job_scheduler",
    "user": "demo",
    "password": "demo"
}

# Redis connection
REDIS_CONFIG = {
    "host": "localhost",
    "port": 6379,
    "decode_responses": True
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

def get_redis():
    return redis.Redis(**REDIS_CONFIG)

# Test connections
try:
    conn = get_db()
    conn.close()
    print("✅ Connected to PostgreSQL")
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")
    print("   Run: docker compose up -d")

try:
    r = get_redis()
    r.ping()
    print("✅ Connected to Redis")
except Exception as e:
    print(f"❌ Redis failed: {e}")
    print("   Run: docker compose up -d")

## 📋 Requirements & Back-of-the-Envelope

### Functional
1. Create a job: **immediate**, **one-time at a date**, or **recurring (cron)**.
2. Execute each due job, retrying on failure up to a limit.
3. Park permanently-failed jobs somewhere a human can look at them (DLQ).
4. Query a job's execution history.

**Out of scope**: job dependencies / DAGs, per-tenant quotas, the task code itself.

### Non-functional
| Requirement | Target | Why |
|---|---|---|
| Throughput | 10k executions/s | Given in the problem statement |
| Timeliness | started within **2 s** of the scheduled time | "Send at 9:00" must not mean 9:04 |
| Delivery guarantee | **At-least-once** | Losing a job is worse than running it twice — *if* tasks are idempotent |
| Durability | A scheduled job survives any single node dying | The scheduler is infrastructure; it cannot lose work |
| Availability | 99.9% | Missed windows are recoverable; permanent loss is not |

The interesting one is the **at-least-once** choice. Exactly-once execution of an
arbitrary side effect is not achievable — you can only get exactly-once *effects*
by making the task idempotent. Notebook 2 demonstrates a real double execution
and then fixes it, rather than hand-waving this.

In [ ]:
# ── Back-of-the-envelope ────────────────────────────────────────────────
PEAK_EXECUTIONS_PER_S = 10_000
AVG_EXECUTIONS_PER_S  = 1_000     # peak is bursty; the average is far lower
EXECUTION_ROW_BYTES   = 300       # ids + status + 3 timestamps + small JSONB
QUEUE_ENTRY_BYTES     = 200       # ZSET member + score + Redis overhead
DB_POLL_WINDOW_S      = 300       # phase 1: pull the next 5 minutes of work
AVG_JOB_DURATION_S    = 0.2
SLOTS_PER_WORKER_BOX  = 50        # concurrent in-flight jobs per machine
SEC_PER_DAY           = 86_400

executions_per_day = AVG_EXECUTIONS_PER_S * SEC_PER_DAY
storage_per_day    = executions_per_day * EXECUTION_ROW_BYTES

# Phase 1 hands the next DB_POLL_WINDOW_S seconds of work to Redis.
rows_per_poll  = AVG_EXECUTIONS_PER_S * DB_POLL_WINDOW_S
queue_bytes    = rows_per_poll * QUEUE_ENTRY_BYTES

# Little's law: in-flight work = arrival rate x service time.
concurrent_jobs_peak = PEAK_EXECUTIONS_PER_S * AVG_JOB_DURATION_S
worker_boxes         = concurrent_jobs_peak / SLOTS_PER_WORKER_BOX

print("📐 Back-of-the-envelope")
print("=" * 70)
print(f"  Executions:            {AVG_EXECUTIONS_PER_S:>12,} /s avg   "
      f"{PEAK_EXECUTIONS_PER_S:>10,} /s peak")
print(f"  Executions per day:    {executions_per_day:>12,.0f}")
print(f"  Execution-row storage: {storage_per_day / 1e9:>12,.1f} GB/day   "
      f"{storage_per_day * 365 / 1e12:>6,.1f} TB/year")
print()
print(f"  Phase-1 poll pulls:    {rows_per_poll:>12,.0f} rows every {DB_POLL_WINDOW_S}s")
print(f"  Redis queue size:      {queue_bytes / 1e6:>12,.0f} MB   ← trivially fits in RAM")
print()
print(f"  Concurrent jobs at peak (Little's law): {concurrent_jobs_peak:>10,.0f}")
print(f"  Worker boxes at {SLOTS_PER_WORKER_BOX} slots each:            {worker_boxes:>10,.0f}")

### What those numbers decide

- **~26 GB of execution rows per day, ~9.5 TB/year.** The `executions` table is
  the thing that will kill you, not the queue. It needs time-based partitioning
  and a retention policy from day one. Cost of that decision: "show me last
  year's runs" becomes an archive query, not a `SELECT`.
- **Only ~60 MB of queue.** The hot path fits in Redis with room to spare. This
  is why the two-phase design works: Postgres is the durable record, Redis holds
  only the next few minutes.
- **300,000 rows per DB poll.** That's why the poll is a *windowed, indexed*
  query (`WHERE status='PENDING' AND scheduled_at < now()+5min`) and not a table
  scan, and why the partial index in `init.sql` exists.
- **2,000 concurrent jobs at peak → ~40 worker boxes.** Workers are stateless
  and horizontal. Nothing about the design changes if that number is 400.

**The timeliness requirement is what forces two phases.** Polling Postgres every
2 seconds to hit a 2-second SLA would mean 43,200 scans a day over a growing
table. Polling every 5 minutes and letting Redis handle the last 300 seconds
costs one extra moving part and buys three orders of magnitude fewer DB scans.
The price: a job created *inside* the current 5-minute window needs a direct
enqueue path, or it will be late.

## 🧱 Core Entities: Task vs Job vs Execution

Before writing any scheduling logic, let's understand the three core entities:

```
┌─────────────────────────────────────────────────────────────────┐
│  TASK (template)                                                │
│  "Send Email" — reusable definition of work                    │
│                                                                 │
│  ┌──────────────────────┐  ┌──────────────────────┐            │
│  │  JOB (instance)      │  │  JOB (instance)      │            │
│  │  Send email to Bob   │  │  Send email to Alice  │            │
│  │  Every Mon 9 AM      │  │  Once on April 15     │            │
│  │                      │  │                       │            │
│  │  ┌────┐ ┌────┐      │  │  ┌────┐              │            │
│  │  │Exec│ │Exec│ ...  │  │  │Exec│              │            │
│  │  │Mon1│ │Mon2│      │  │  │Apr15│             │            │
│  │  └────┘ └────┘      │  │  └────┘              │            │
│  └──────────────────────┘  └──────────────────────┘            │
└─────────────────────────────────────────────────────────────────┘
```

**Why separate them?** A recurring job ("every day at 10 AM") creates a new execution row *each time* it runs. We need to track each run independently — did Monday's run succeed? Did Tuesday's fail?

Let's look at our database schema.

In [ ]:
# Explore the seed data in our database
conn = get_db()
cursor = conn.cursor()

print("📋 Available Tasks (templates)")
print("=" * 70)
cursor.execute("SELECT id, name, max_retries, timeout_seconds FROM tasks")
for row in cursor.fetchall():
    print(f"  {row[0]:<20} {row[1]:<25} retries={row[2]}  timeout={row[3]}s")

print()
print("📋 Scheduled Jobs (instances)")
print("=" * 70)
cursor.execute("""
    SELECT j.id::text, j.user_id, j.task_id, j.schedule_type, j.schedule_value
    FROM jobs j ORDER BY j.created_at
""")
for row in cursor.fetchall():
    sched = row[4] or 'now'
    print(f"  {row[1]:<15} {row[2]:<20} {row[3]:<12} {sched}")

print()
print("📋 Executions (individual runs)")
print("=" * 70)
cursor.execute("""
    SELECT e.status, e.scheduled_at, e.attempt, e.worker_id, t.name
    FROM executions e
    JOIN jobs j ON e.job_id = j.id
    JOIN tasks t ON j.task_id = t.id
    ORDER BY e.scheduled_at
""")
for row in cursor.fetchall():
    worker = row[3] or 'unassigned'
    print(f"  {row[4]:<25} {row[0]:<12} attempt={row[2]}  worker={worker}")

conn.close()

## 🔴 The Problem: How Does the Scheduler Know What to Run Next?

The naive approach is to poll the database every few seconds:

```sql
SELECT * FROM executions
WHERE status = 'PENDING' AND scheduled_at <= NOW()
ORDER BY scheduled_at
LIMIT 100;
```

This works at small scale, but breaks down at **10,000 jobs/second**:

- Polling every 2 seconds fetches ~20,000 rows each time
- The database is hammered with repeated full scans
- Multiple workers compete for the same rows (race conditions)

**The solution: a two-phase architecture.**

1. **Phase 1 (DB poll)** — Every ~5 minutes, query the database for upcoming jobs and push them into a Redis queue
2. **Phase 2 (Redis queue)** — Workers pull from Redis in real-time with sub-millisecond latency

Let's build Phase 2 first — the Redis priority queue.

## ⚡ Redis Sorted Sets: The Perfect Priority Queue

A Redis **sorted set** (ZSET) stores members with a numeric score and keeps them sorted automatically.

For a job queue, the score is the **scheduled execution timestamp** (Unix epoch). Jobs with earlier timestamps get processed first.

```
ZSET "job_queue"
┌──────────────────────────┬────────────┐
│ Member (execution_id)    │ Score (ts) │
├──────────────────────────┼────────────┤
│ exec-001                 │ 1711900800 │  ← earliest → process first
│ exec-002                 │ 1711900860 │
│ exec-003                 │ 1711900920 │
│ exec-004                 │ 1711901000 │  ← latest → process last
└──────────────────────────┴────────────┘
```

Key operations:
- `ZADD` — add a job with its scheduled time as the score
- `ZRANGEBYSCORE` — get all jobs whose score ≤ now (i.e. ready to run)
- `ZPOPMIN` — atomically remove and return the job with the lowest score

In [ ]:
# Build a simple priority queue using Redis sorted sets

r = get_redis()
QUEUE_KEY = "job_queue:default"

# Clean up from any previous runs
r.delete(QUEUE_KEY)

def enqueue_job(execution_id: str, scheduled_at: float, metadata: dict):
    """Add a job to the priority queue.
    
    The score is the scheduled timestamp — lower scores run first.
    The member is a JSON blob with the execution details.
    """
    member = json.dumps({
        "execution_id": execution_id,
        "metadata": metadata
    })
    r.zadd(QUEUE_KEY, {member: scheduled_at})

def dequeue_ready_jobs(limit: int = 10) -> list:
    """Get jobs that are ready to execute (scheduled_at <= now).
    
    Uses ZPOPMIN-style logic: fetch jobs with score <= current time,
    then remove them from the queue atomically.
    """
    now = time.time()
    # Get jobs ready to run (score <= now)
    ready = r.zrangebyscore(QUEUE_KEY, "-inf", now, start=0, num=limit)
    
    jobs = []
    for member in ready:
        # Remove from queue (atomic: only one worker gets it)
        removed = r.zrem(QUEUE_KEY, member)
        if removed:  # we won the race
            jobs.append(json.loads(member))
    return jobs

def peek_queue() -> list:
    """Look at the queue without removing anything."""
    items = r.zrange(QUEUE_KEY, 0, -1, withscores=True)
    return [(json.loads(member), score) for member, score in items]

print("✅ Priority queue functions defined")
print("   enqueue_job(execution_id, scheduled_at, metadata)")
print("   dequeue_ready_jobs(limit)")
print("   peek_queue()")

In [ ]:
# Let's add some jobs with different scheduled times

now = time.time()

# Jobs scheduled in the past (ready to run immediately)
enqueue_job("exec-001", now - 60, {"task": "send_email", "to": "alice@example.com"})
enqueue_job("exec-002", now - 30, {"task": "send_email", "to": "bob@example.com"})

# Job scheduled for right now
enqueue_job("exec-003", now, {"task": "generate_report", "type": "daily_sales"})

# Jobs scheduled in the future (not ready yet)
enqueue_job("exec-004", now + 300, {"task": "cleanup_temp"})    # 5 min from now
enqueue_job("exec-005", now + 3600, {"task": "backup_database"}) # 1 hour from now

print("📋 Queue contents (sorted by scheduled time):")
print("=" * 65)
for item, score in peek_queue():
    delta = score - now
    if delta < 0:
        timing = f"{abs(delta):.0f}s AGO (overdue!)"
    elif delta < 1:
        timing = "NOW"
    else:
        timing = f"in {delta:.0f}s"
    print(f"  {item['execution_id']}  score={score:.0f}  {timing}")
    print(f"           metadata={item['metadata']}")

In [ ]:
# Now dequeue only the jobs that are READY (scheduled_at <= now)

ready_jobs = dequeue_ready_jobs(limit=10)

print(f"🚀 Dequeued {len(ready_jobs)} ready jobs:")
for job in ready_jobs:
    print(f"  {job['execution_id']}  →  {job['metadata']}")

print()
print(f"📋 Remaining in queue: {r.zcard(QUEUE_KEY)} jobs")
for item, score in peek_queue():
    delta = score - time.time()
    print(f"  {item['execution_id']}  (runs in {delta:.0f}s)")

print()
print("💡 Only past-due and current jobs were dequeued.")
print("   Future jobs stay in the queue until their time comes.")

## 🎯 Adding Priority Levels

Not all jobs are equal. A password reset email should run before a weekly analytics report.

We can handle this with **multiple queues** — one per priority level:

```
job_queue:critical   ← checked first  (password resets, alerts)
job_queue:high       ← checked second (user-facing emails)
job_queue:normal     ← checked third  (reports, syncs)
job_queue:low        ← checked last   (cleanup, backups)
```

Workers always check higher-priority queues first. But this creates a risk: **starvation**. If the critical queue always has jobs, low-priority jobs never run.

We'll implement **weighted fair queuing** to prevent this.

In [ ]:
import random

# Priority queue system with multiple levels
PRIORITIES = ["critical", "high", "normal", "low"]
PRIORITY_QUEUES = {p: f"job_queue:{p}" for p in PRIORITIES}

# Weighted fair queuing: how many jobs to pull from each level per cycle
# Critical gets 4x the share of low-priority
PRIORITY_WEIGHTS = {
    "critical": 8,   # 8 out of 16 slots → 50%
    "high":     4,   # 4 out of 16 slots → 25%
    "normal":   3,   # 3 out of 16 slots → ~19%
    "low":      1,   # 1 out of 16 slots → ~6%
}

# Clean up
for key in PRIORITY_QUEUES.values():
    r.delete(key)

def enqueue_priority_job(execution_id: str, scheduled_at: float, 
                         priority: str, metadata: dict):
    """Add a job to the appropriate priority queue."""
    queue_key = PRIORITY_QUEUES[priority]
    member = json.dumps({
        "execution_id": execution_id,
        "priority": priority,
        "metadata": metadata
    })
    r.zadd(queue_key, {member: scheduled_at})

def dequeue_with_fair_scheduling(batch_size: int = 16) -> list:
    """Dequeue jobs using weighted fair scheduling.
    
    Each priority level gets a proportional share of the batch.
    This prevents starvation — low-priority jobs always get SOME slots.
    """
    now = time.time()
    total_weight = sum(PRIORITY_WEIGHTS.values())
    all_jobs = []
    
    for priority in PRIORITIES:
        # Calculate how many slots this priority gets
        slots = max(1, int(batch_size * PRIORITY_WEIGHTS[priority] / total_weight))
        queue_key = PRIORITY_QUEUES[priority]
        
        # Get ready jobs from this priority queue
        ready = r.zrangebyscore(queue_key, "-inf", now, start=0, num=slots)
        for member in ready:
            if r.zrem(queue_key, member):
                all_jobs.append(json.loads(member))
    
    return all_jobs

print("✅ Priority queue system defined")
print(f"   Queues: {list(PRIORITY_QUEUES.values())}")
print(f"   Weights: {PRIORITY_WEIGHTS}")

In [ ]:
# Simulate a burst of mixed-priority jobs

now = time.time()

# Add 5 jobs at each priority level, all ready to run
job_types = {
    "critical": "password_reset",
    "high":     "welcome_email",
    "normal":   "daily_report",
    "low":      "cleanup_temp",
}

for priority, task in job_types.items():
    for i in range(5):
        enqueue_priority_job(
            execution_id=f"{priority}-{i+1}",
            scheduled_at=now - random.uniform(0, 60),  # all overdue
            priority=priority,
            metadata={"task": task, "seq": i+1}
        )

print("📋 Queue sizes after adding 5 jobs per priority:")
for priority in PRIORITIES:
    count = r.zcard(PRIORITY_QUEUES[priority])
    print(f"  {priority:<10} {count} jobs")

print()

# Dequeue one batch of 16
batch = dequeue_with_fair_scheduling(batch_size=16)

print(f"🚀 Dequeued {len(batch)} jobs with fair scheduling:")
counts = {}
for job in batch:
    p = job['priority']
    counts[p] = counts.get(p, 0) + 1
    print(f"  [{p:<10}] {job['execution_id']}  →  {job['metadata']['task']}")

print()
print("📊 Distribution in this batch:")
for p in PRIORITIES:
    c = counts.get(p, 0)
    bar = "█" * c
    print(f"  {p:<10} {c} jobs  {bar}")

print()
print("📋 Remaining in queues:")
for priority in PRIORITIES:
    count = r.zcard(PRIORITY_QUEUES[priority])
    print(f"  {priority:<10} {count} jobs")

print()
print("💡 Critical jobs got the most slots, but low-priority jobs still got served!")
print("   This prevents starvation while respecting priority.")

## 🔄 Phase 1: Loading Jobs from the Database into Redis

In production, a **watcher process** runs every ~5 minutes and queries the database for upcoming jobs, then pushes them into the Redis queues.

```
┌──────────┐    every 5 min    ┌──────────┐    workers pull    ┌──────────┐
│ Postgres │  ─────────────►  │  Redis   │  ──────────────►  │ Workers  │
│ (durable)│   watcher poll    │ (fast)   │   real-time        │ (execute)│
└──────────┘                   └──────────┘                    └──────────┘
```

Why two phases?
- **Postgres** = durable storage — jobs survive crashes, can be queried by users
- **Redis** = fast queue — sub-millisecond dequeue, perfect for 10k jobs/sec

In [ ]:
# Simulate the watcher process: poll DB → push to Redis

def watcher_poll(look_ahead_minutes: int = 5):
    """Poll the database for pending executions and push them to Redis.
    
    In production this runs on a timer (every ~5 minutes).
    It finds all PENDING executions scheduled within the look-ahead window
    and pushes them into the appropriate Redis priority queue.
    """
    conn = get_db()
    cursor = conn.cursor()
    
    # Find pending executions due in the next N minutes
    cursor.execute("""
        SELECT e.id, e.job_id, e.scheduled_at, j.task_id, j.parameters
        FROM executions e
        JOIN jobs j ON e.job_id = j.id
        WHERE e.status = 'PENDING'
          AND e.scheduled_at <= NOW() + INTERVAL '%s minutes'
        ORDER BY e.scheduled_at
    """, (look_ahead_minutes,))
    
    rows = cursor.fetchall()
    enqueued = 0
    
    for row in rows:
        exec_id, job_id, scheduled_at, task_id, params = row
        
        # Convert scheduled_at to Unix timestamp for the ZSET score
        score = scheduled_at.timestamp()
        
        # Determine priority (in production this would come from job metadata)
        priority = "normal"
        
        # Push to Redis
        queue_key = PRIORITY_QUEUES[priority]
        member = json.dumps({
            "execution_id": str(exec_id),
            "job_id": str(job_id),
            "task_id": task_id,
            "parameters": params
        })
        r.zadd(queue_key, {member: score})
        
        # Mark as QUEUED in the database so we don't enqueue it again
        cursor.execute(
            "UPDATE executions SET status = 'QUEUED' WHERE id = %s AND status = 'PENDING'",
            (exec_id,)
        )
        enqueued += 1
    
    conn.commit()
    conn.close()
    return enqueued

# Run the watcher poll
count = watcher_poll(look_ahead_minutes=60 * 24 * 7)  # look far ahead for demo
print(f"🔍 Watcher polled database: found and enqueued {count} pending executions")

print()
print("📋 Redis queue state after poll:")
for priority in PRIORITIES:
    count = r.zcard(PRIORITY_QUEUES[priority])
    if count > 0:
        print(f"  {priority}: {count} jobs")

## ❌ Bad Practice vs. ✅ Best Practice: Polling Postgres vs. Dequeuing from Redis

Before moving on, let us make the "two-phase architecture" concrete by
measuring it. We will have workers "dequeue" the same 2,000 jobs two ways:

1. **Naive (bad):** each poll runs `SELECT ... WHERE status='PENDING'
   ORDER BY scheduled_at LIMIT 100` and then `UPDATE` to mark rows `QUEUED`.
2. **Two-phase (best):** the rows are already in a Redis ZSET; workers use
   `ZPOPMIN` to grab the next batch.

This is a direct bad-to-best demonstration of why schedulers at scale keep a
fast queue in front of their durable store.


In [ ]:
# Create 2,000 pending executions for the benchmark.
# We attach them to an existing seed job so foreign keys are satisfied.

import time

conn = get_db()
cursor = conn.cursor()

# Pick any existing job to associate these benchmark executions with.
cursor.execute("SELECT id FROM jobs LIMIT 1")
bench_job_id = cursor.fetchone()[0]

# Clear any leftovers from previous runs of this cell.
cursor.execute("DELETE FROM executions WHERE worker_id = 'bench-naive'")

# Insert 2,000 PENDING executions scheduled in the past (ready now).
cursor.execute(
    """
    INSERT INTO executions (job_id, status, scheduled_at, worker_id)
    SELECT %s, 'PENDING', NOW() - (s || ' seconds')::interval, 'bench-naive'
    FROM generate_series(1, 2000) AS s
    """,
    (bench_job_id,),
)
conn.commit()

# ---- Naive approach: poll Postgres, update rows in batches of 100 ----
start = time.time()
total = 0
while True:
    cursor.execute(
        """
        UPDATE executions
        SET status = 'QUEUED'
        WHERE id IN (
            SELECT id FROM executions
            WHERE status = 'PENDING' AND worker_id = 'bench-naive'
            ORDER BY scheduled_at
            LIMIT 100
            FOR UPDATE SKIP LOCKED
        )
        RETURNING id
        """
    )
    rows = cursor.fetchall()
    conn.commit()
    if not rows:
        break
    total += len(rows)
naive_time = time.time() - start

print(f"Naive (Postgres poll+update): {total} jobs in {naive_time*1000:.0f} ms "
      f"-> {total/naive_time:,.0f} jobs/sec")

# ---- Two-phase approach: same 2,000 jobs preloaded into a Redis ZSET ----
BENCH_ZSET = "job_queue:bench_compare"
r.delete(BENCH_ZSET)

pipe = r.pipeline()
now = time.time()
for i in range(2000):
    pipe.zadd(BENCH_ZSET, {f"exec-{i}": now - 1})
pipe.execute()

start = time.time()
total = 0
while True:
    popped = r.zpopmin(BENCH_ZSET, count=100)
    if not popped:
        break
    total += len(popped)
redis_time = time.time() - start

print(f"Two-phase (Redis ZPOPMIN):    {total} jobs in {redis_time*1000:.0f} ms "
      f"-> {total/redis_time:,.0f} jobs/sec")

print()
print(f"Redis is ~{naive_time/redis_time:.1f}x faster here, and the gap widens")
print("as the executions table grows and multiple workers compete for rows.")

# Clean up benchmark rows so we do not pollute later cells.
cursor.execute("DELETE FROM executions WHERE worker_id = 'bench-naive'")
conn.commit()
conn.close()


## 📊 Measuring Queue Performance

Let's benchmark our Redis-based priority queue to see how it performs under load.

In [ ]:
# Benchmark: enqueue and dequeue 10,000 jobs

BENCH_KEY = "job_queue:benchmark"
r.delete(BENCH_KEY)

NUM_JOBS = 10_000
now = time.time()

# Enqueue benchmark
start = time.time()
pipe = r.pipeline()  # pipeline batches commands for much better throughput
for i in range(NUM_JOBS):
    member = json.dumps({"execution_id": f"bench-{i}", "task": "noop"})
    score = now - random.uniform(0, 300)  # all overdue
    pipe.zadd(BENCH_KEY, {member: score})
pipe.execute()
enqueue_time = time.time() - start

print(f"📥 Enqueued {NUM_JOBS:,} jobs in {enqueue_time:.2f}s")
print(f"   Throughput: {NUM_JOBS/enqueue_time:,.0f} enqueues/sec")
print()

# Dequeue benchmark
start = time.time()
dequeued = 0
while True:
    results = r.zpopmin(BENCH_KEY, count=100)  # pop 100 at a time
    if not results:
        break
    dequeued += len(results)
dequeue_time = time.time() - start

print(f"📤 Dequeued {dequeued:,} jobs in {dequeue_time:.2f}s")
print(f"   Throughput: {dequeued/dequeue_time:,.0f} dequeues/sec")
print()
print("💡 Redis sorted sets can handle tens of thousands of operations per second.")
print("   This is why they're perfect for job scheduling priority queues.")

## 🧹 Cleanup

In [ ]:
# Clean up all Redis keys we created
r = get_redis()
for key in r.keys("job_queue:*"):
    r.delete(key)
print("🧹 Cleaned up Redis keys")

# Reset execution statuses back to PENDING
conn = get_db()
cursor = conn.cursor()
cursor.execute("UPDATE executions SET status = 'PENDING' WHERE status = 'QUEUED'")
conn.commit()
conn.close()
print("🧹 Reset execution statuses")

## 📚 Summary

### Key Takeaways

1. **Task → Job → Execution** — separate the template from the schedule from the individual run
2. **Redis sorted sets** are the ideal data structure for priority queues — O(log N) insert, O(1) pop-min
3. **Two-phase architecture** — database for durability, Redis for speed
4. **Weighted fair queuing** prevents starvation while respecting priority levels
5. **Redis can handle 10k+ ops/sec** easily, making it perfect for high-throughput scheduling

### Interview Tips

- Always mention the job/execution separation for recurring jobs
- Use `ZPOPMIN` or `ZRANGEBYSCORE` + `ZREM` for atomic dequeue
- Multiple priority queues > single queue with priority scores (simpler, avoids rebalancing)

### Next Up

In **Notebook 2**, we'll build the **distributed worker pool** — handling concurrent execution, visibility timeouts, retries, and at-least-once delivery.